# scGPT embedding arm

Compare PCA vs scGPT zero-shot embeddings on one accession. Run on the Lund bioinformatics server (CPU-only).

Prerequisite: download the scGPT whole-human checkpoint (`args.json`, `vocab.json`, `best_model.pt`) to `models/scgpt/whole_human/`.

In [ ]:
from pathlib import Path

import pandas as pd

from cluster_validation import ClusterValidationConfig, run_cluster_validation
from cluster_validation.cell_type_metrics import compute_nse_kld_row
from cluster_validation.viz import plot_all
from shared.repo import REPO_ROOT

ROOT = REPO_ROOT

## Config

In [ ]:
SRX = "SRX12366723"
H5AD_DIR = ROOT / "data/scbasecount/2026-01-12/h5ad/GeneFull/Homo_sapiens"
SUMMARY = ROOT / "output/metadata/datasets.csv"
MODEL_DIR = ROOT / "models/scgpt/whole_human"
EMBED_DIR = ROOT / "output/scgpt_embed"
OUTPUT_DIR = ROOT / "tmp/scgpt_embedding/data"
FIGS_DIR = ROOT / "tmp/scgpt_embedding/figs"

RAW_H5AD = H5AD_DIR / f"{SRX}.h5ad"
EMBED_PATH = EMBED_DIR / f"{SRX}_scgpt.npz"

## Generate scGPT embeddings (isolated env)

In [ ]:
EMBED_DIR.mkdir(parents=True, exist_ok=True)

cmd = (
    f"uv run --directory {ROOT / 'tools/scgpt_embed'} python embed.py "
    f"--h5ad {RAW_H5AD} "
    f"--model-dir {MODEL_DIR} "
    f"--out {EMBED_PATH}"
)
print(cmd)
!{cmd}

## Run PCA baseline

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pca_cfg = ClusterValidationConfig(
    srxAccession=SRX,
    summaryPath=SUMMARY,
    localH5adRoot=H5AD_DIR,
    outputDir=OUTPUT_DIR / "pca",
    embedding="pca",
)
adata_pca, result_pca = run_cluster_validation(pca_cfg)
plot_all(adata_pca, result_pca, figs_dir=FIGS_DIR / SRX / "pca")

## Run scGPT arm

In [ ]:
scgpt_cfg = ClusterValidationConfig(
    srxAccession=SRX,
    summaryPath=SUMMARY,
    localH5adRoot=H5AD_DIR,
    outputDir=OUTPUT_DIR / "scgpt",
    embedding="scgpt",
    scgptEmbedPath=EMBED_PATH,
)
adata_scgpt, result_scgpt = run_cluster_validation(scgpt_cfg)
plot_all(adata_scgpt, result_scgpt, figs_dir=FIGS_DIR / SRX / "scgpt")

## Compare STATE vs Leiden agreement

In [ ]:
def state_leiden_table(adata, merged_key: str) -> pd.DataFrame:
    return pd.crosstab(adata.obs["cell_type"], adata.obs[merged_key])


pca_nse, pca_kld = compute_nse_kld_row(adata_pca, result_pca.mergedKey)
scgpt_nse, scgpt_kld = compute_nse_kld_row(adata_scgpt, result_scgpt.mergedKey)

comparison = pd.DataFrame(
    {
        "pca_nse": pd.Series(pca_nse),
        "scgpt_nse": pd.Series(scgpt_nse),
        "pca_kld": pd.Series(pca_kld),
        "scgpt_kld": pd.Series(scgpt_kld),
    }
).sort_values("pca_nse")

display(comparison.head(15))
display(state_leiden_table(adata_pca, result_pca.mergedKey).iloc[:10, :10])
display(state_leiden_table(adata_scgpt, result_scgpt.mergedKey).iloc[:10, :10])

In [ ]:
summary = pd.DataFrame(
    [
        {
            "embedding": "pca",
            "selected_resolution": result_pca.selectedResolution,
            "n_clusters": result_pca.nClustersPostMerge,
            "n_pcs": result_pca.nPcs,
        },
        {
            "embedding": "scgpt",
            "selected_resolution": result_scgpt.selectedResolution,
            "n_clusters": result_scgpt.nClustersPostMerge,
            "n_pcs": result_scgpt.nPcs,
        },
    ]
)
summary